Intellisense using GPT-3

# Training Data Generation

## Installing OpenAI Package

This cell installs the OpenAI Python package, which allows us to interact with OpenAI's GPT models for code completion and other language tasks.

In [73]:
%pip install openai


7176.07s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Importing the OpenAI library

We import the OpenAI library to use its API for generating code completions and other language-based tasks.

In [74]:
import openai

## Creating a list

Here, we define a list called `code` that contains three Python import statements. These will be used as sample code lines for generating training data.

In [75]:
code=['import numpy as np',
'import pandas as pd',
'import matplotlib.pyplot as plt',]

## Generating Training Data

In this section, we generate training data for code completion.  
For each code line in the `code` list, we create multiple prefix-suffix pairs starting from the 8th character.  
Each pair consists of:
- `A:` The prefix (the code up to a certain character)
- `B:` The suffix (the remaining code from that character onward)

This format is useful for training or evaluating models that predict code completions based on a given prefix.

In [76]:
training_data = []
for sent in code :
    # Start from the 8th character (index 7) to the end of the string
    for i in range(7, len(sent)):
        loc_data = []
        # Create a prefix (up to i+1) and a suffix (from i onward)
        loc_data.append(sent[:i+1])  # Prefix: code up to current character
        loc_data.append(sent[i:])    # Suffix: code from current character to end
        training_data.append(loc_data)  # Add the pair to the training data

# Print the first 5 prefix-suffix pairs for inspection
count = 0
for train_set in training_data:
    print('A:', train_set[0])  # Print the prefix
    print('B:', train_set[1])  # Print the suffix
    count += 1
    #if count == 5:
    #    break

A: import n
B: numpy as np
A: import nu
B: umpy as np
A: import num
B: mpy as np
A: import nump
B: py as np
A: import numpy
B: y as np
A: import numpy 
B:  as np
A: import numpy a
B: as np
A: import numpy as
B: s np
A: import numpy as 
B:  np
A: import numpy as n
B: np
A: import numpy as np
B: p
A: import p
B: pandas as pd
A: import pa
B: andas as pd
A: import pan
B: ndas as pd
A: import pand
B: das as pd
A: import panda
B: as as pd
A: import pandas
B: s as pd
A: import pandas 
B:  as pd
A: import pandas a
B: as pd
A: import pandas as
B: s pd
A: import pandas as 
B:  pd
A: import pandas as p
B: pd
A: import pandas as pd
B: d
A: import m
B: matplotlib.pyplot as plt
A: import ma
B: atplotlib.pyplot as plt
A: import mat
B: tplotlib.pyplot as plt
A: import matp
B: plotlib.pyplot as plt
A: import matpl
B: lotlib.pyplot as plt
A: import matplo
B: otlib.pyplot as plt
A: import matplot
B: tlib.pyplot as plt
A: import matplotl
B: lib.pyplot as plt
A: import matplotli
B: ib.pyplot as plt
A: impo

# Functions & Classes

## Defining the Response Function

The next cell defines a function called `get_response` that uses OpenAI's API to generate a text response based on a provided prompt. This function will later be used to get code completions.

In [77]:
'''
def get_response(final_prompt):
    response = openai.Completion.create(
        model='text-davinci-003',
        prompt=final_prompt,
        temperature=0.7,
        max_tokens=128,
        top_p=.53,
        frequency_penalty=0.02,
        presence_penalty=0.98
    )
    return (response['choices'][0]['text'])

The latest python library doesn't support the API and text-davinci-003 model.
Hence, using new model and way.
'''

def get_response(final_prompt):
    response = openai.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user", "content": final_prompt}
        ],
        temperature=0.7,
        max_tokens=128,
        top_p=0.53,
        frequency_penalty=0.02,
        presence_penalty=0.98
    )
    return response.choices[0].message.content

## Preparing the Examples Class

In this section, we define the `examples` class. This class is designed to store training examples (prefix-suffix pairs) and to build a final prompt that can be used to generate completions. The methods within this class help manage these examples and prepare the prompt format.

In [78]:
import re # importing the regular expression (regex) module to use regex functions for text processing
class examples():
    def __init__(self, prefix, suffix):
        self.training_examples = []
        self.final_prompt_str = ''
        self.seperator = '\n'
        self.in_sep = 'A:'
        self.out_sep = 'B:'
    def add(self, training_input, training_output):
        train_str=self.in_sep + training_input + self.seperator + self.out_sep + training_output + self.seperator
        self.training_examples.append(train_str)
    def add_prompt(self, prompt):
        self.final_prompt_str = self.get_examples() + self.in_sep + prompt + self.seperator
        #print(self.final_prompt_str)
    def get_final_prompt(self):
        return self.final_prompt_str + self.out_sep
        #text_after = re.sub(regex_search_term, regex_replacement, text_before)
    def get_examples(self):
        example_str = ''
        for exmpl in self.training_examples:
            example_str += exmpl
        return example_str

## Initializing the Examples Instance

This cell attempts to initialize an instance of the `examples` class. If an instance already exists, it is deleted before creating a new one. This ensures that you start with a fresh set of training examples.

In [79]:
try:
    del train_examples
    train_examples = examples(prefix="your_prefix_value", suffix="your_suffix_value")
except Exception:
    train_examples = examples(prefix="your_prefix_value", suffix="your_suffix_value")

## Adding Training Data to the Examples Instance

Here, we loop through the generated training data (prefix-suffix pairs) and add each pair to the `train_examples` object using the `add` method. This prepares all of our training examples for constructing the final prompt for code completion.

In [80]:
count = 0
for train_set in training_data:
    train_examples.add(train_set[0], train_set[1])
    count += 1
    #if count > 5:
        #break

## Preview Final Prompt

In this cell, we display the final prompt generated by combining all our training examples. This preview helps verify that our training data has been correctly processed and concatenated into the final prompt structure, which will be used for generating code completions.

In [82]:
train_examples.get_examples()

'A:import n\nB:numpy as np\nA:import nu\nB:umpy as np\nA:import num\nB:mpy as np\nA:import nump\nB:py as np\nA:import numpy\nB:y as np\nA:import numpy \nB: as np\nA:import numpy a\nB:as np\nA:import numpy as\nB:s np\nA:import numpy as \nB: np\nA:import numpy as n\nB:np\nA:import numpy as np\nB:p\nA:import p\nB:pandas as pd\nA:import pa\nB:andas as pd\nA:import pan\nB:ndas as pd\nA:import pand\nB:das as pd\nA:import panda\nB:as as pd\nA:import pandas\nB:s as pd\nA:import pandas \nB: as pd\nA:import pandas a\nB:as pd\nA:import pandas as\nB:s pd\nA:import pandas as \nB: pd\nA:import pandas as p\nB:pd\nA:import pandas as pd\nB:d\nA:import m\nB:matplotlib.pyplot as plt\nA:import ma\nB:atplotlib.pyplot as plt\nA:import mat\nB:tplotlib.pyplot as plt\nA:import matp\nB:plotlib.pyplot as plt\nA:import matpl\nB:lotlib.pyplot as plt\nA:import matplo\nB:otlib.pyplot as plt\nA:import matplot\nB:tlib.pyplot as plt\nA:import matplotl\nB:lib.pyplot as plt\nA:import matplotli\nB:ib.pyplot as plt\nA:impo

# Testing with New Inputs

## Enter Your Custom Prompt

Provide a custom code snippet or module import here to generate completions using GPT-3.

In [83]:
prompt = 'import stats'
train_examples.add_prompt(prompt)
#final_prompt = train_examples.get_final_prompt()
#print(final_prompt)
train_examples.get_final_prompt()

'A:import n\nB:numpy as np\nA:import nu\nB:umpy as np\nA:import num\nB:mpy as np\nA:import nump\nB:py as np\nA:import numpy\nB:y as np\nA:import numpy \nB: as np\nA:import numpy a\nB:as np\nA:import numpy as\nB:s np\nA:import numpy as \nB: np\nA:import numpy as n\nB:np\nA:import numpy as np\nB:p\nA:import p\nB:pandas as pd\nA:import pa\nB:andas as pd\nA:import pan\nB:ndas as pd\nA:import pand\nB:das as pd\nA:import panda\nB:as as pd\nA:import pandas\nB:s as pd\nA:import pandas \nB: as pd\nA:import pandas a\nB:as pd\nA:import pandas as\nB:s pd\nA:import pandas as \nB: pd\nA:import pandas as p\nB:pd\nA:import pandas as pd\nB:d\nA:import m\nB:matplotlib.pyplot as plt\nA:import ma\nB:atplotlib.pyplot as plt\nA:import mat\nB:tplotlib.pyplot as plt\nA:import matp\nB:plotlib.pyplot as plt\nA:import matpl\nB:lotlib.pyplot as plt\nA:import matplo\nB:otlib.pyplot as plt\nA:import matplot\nB:tlib.pyplot as plt\nA:import matplotl\nB:lib.pyplot as plt\nA:import matplotli\nB:ib.pyplot as plt\nA:impo

## Final Prompt Preview

Below is the final prompt constructed from all the training examples along with your custom input.

In [84]:
def process_request(prompt):
    train_examples.add_prompt(prompt)
    final_prompt = train_examples.get_final_prompt()
    #print(final_prompt)
    response = get_response(final_prompt)
    #print(response)
    print('-------------------------------')
    intellisense = []
    text_after = response.split('\n')
    for i, intelli in enumerate(text_after):
        if i % 2 == 0:
            intellisense.append(intelli[2:])
            #print(intelli)
    for p in set(intellisense):
        print(p)

## Code Completion Output

This section displays the code completions generated by the GPT-3 model based on your input prompt.

In [88]:
prompt = input()
#prompt = 'import p'
print('-------------------------------')
process_request(prompt)

-------------------------------


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable